# درس ۰۳ - الگوهای طراحی عامل محور

در این درس، سه الگوی طراحی پایه برای ساخت عامل‌های هوش مصنوعی مؤثر را بررسی می‌کنیم:

1. **دستورات واضح برای عامل** — ساخت درخواست‌های دقیق و تعریف‌کننده نقش که رفتار عامل را هدایت می‌کنند  
2. **خروجی ساختاریافته با مدل‌های Pydantic** — اطمینان از اینکه عامل‌ها داده‌های پیش‌بینی‌پذیر و اعتبارسنجی‌شده بازمی‌گردانند  
3. **عامل‌های با مسئولیت واحد** — طراحی عامل‌های متمرکز که هرکدام یک کار را به خوبی انجام می‌دهند

ما هر الگو را روی سناریوی **توصیه‌گر مقصد سفر** اعمال خواهیم کرد و به‌صورت مرحله‌به‌مرحله سیستمی می‌سازیم که بتواند مقصدها را پیشنهاد دهد، دسترسی را بررسی کند و امور لجستیکی را مدیریت کند.


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## الگو ۱: دستورالعمل‌های واضح برای نماینده

تأثیرگذارترین الگو، ساده‌ترین آن نیز هست: نوشتن دستورالعمل‌های واضح و دقیق برای نماینده شما.

دستورالعمل‌های خوب مشخص می‌کنند:
- **چه کسی** نماینده است (شخصیت و لحن)
- **چه کاری** باید انجام دهد (وظایف گام به گام)
- **چگونه** باید رفتار کند (محدودیت‌ها و سبک)

در زیر، یک نماینده متصدی سفر ایجاد می‌کنیم با دستورالعمل‌های صریح که هر پاسخی که تولید می‌کند را شکل می‌دهد.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## الگو ۲: خروجی ساختاریافته با مدل‌های Pydantic

متن آزاد برای گفتگو کاربردی است، اما سیستم‌های پایین‌دستی به داده‌های ساختاریافته نیاز دارند.  
با جفت‌کردن **مدل‌های Pydantic** با یک **تابع ابزار**، ما می‌توانیم:

- یک طرحواره دقیق برای خروجی عامل تعریف کنیم  
- پاسخ‌ها را به‌طور خودکار اعتبارسنجی کنیم  
- نتایج عامل را به‌طور قابل اطمینان در منطق برنامه ادغام کنیم  

ما همچنین ابزاری معرفی می‌کنیم که جزئیات مقصد را بازمی‌گرداند تا عامل توصیه‌های خود را بر داده‌های واقعی پایه‌گذاری کند.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## الگو ۳: نمایندگان مسئولیت تک

وظایف پیچیده با تقسیم کار بین چند نماینده متمرکز که هرکدام یک مسئولیت دارند، بهتر انجام می‌شود:

- یک **کارشناس مقصد** که درباره مکان‌ها و دسترسی‌ها اطلاع دارد
- یک **برنامه‌ریز لجستیک** که پروازها، هتل‌ها و برنامه‌های سفر را مدیریت می‌کند

این مشابه اصل مهندسی نرم‌افزار *تفکیک وظایف* است — هر نماینده به‌صورت مستقل آزمایش، نگهداری و بهبود می‌یابد.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## خلاصه

در این درس سه الگوی طراحی عامل‌محور را در سناریوی توصیه‌گر سفر به کار بردیم:

| الگو | ایده کلیدی | مزیت |
|---|---|---|
| **دستورالعمل‌های روشن** | تعریف پرسونای عامل، مسئولیت‌ها و محدودیت‌ها از ابتدا | رفتار عامل سازگار و مطابق با برند |
| **خروجی ساختاریافته** | استفاده از مدل‌های Pydantic به عنوان قالب پاسخ | نتایج معتبر و قابل‌خواندن توسط ماشین |
| **تک مسئولیتی** | اختصاص یک وظیفه متمرکز به هر عامل | آسان‌تر برای تست، نگهداری و ترکیب |

این الگوها به صورت طبیعی ترکیب می‌شوند — می‌توانید دستورالعمل‌های روشن را با خروجی ساختاریافته در یک عامل با تک‌مسئولیتی ترکیب کنید تا سیستم‌هایی مقاوم و آماده تولید بسازید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
